In [1]:
# ============================================================
# CELL 1: Imports and global config
# ============================================================

from pathlib import Path
import gc
import json
import time
from datetime import datetime
from typing import Dict, List

import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    Qwen2AudioForConditionalGeneration,
    BitsAndBytesConfig,
)

from comet import download_model, load_from_checkpoint


PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
OUTPUTS_ROOT = PROJECT_ROOT / "outputs"
RESULTS_ROOT = OUTPUTS_ROOT / "experiment_results"
REGISTRY_ROOT = OUTPUTS_ROOT / "results_registry"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
REGISTRY_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
DEVICE = "cuda:0"
PROMPT_TEXT = "Translate this English speech into Chinese."
MAX_NEW_TOKENS = 64
COMET_BATCH_SIZE = 16

DEV_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_dev_en_zh.csv"
EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

# ----------------------------
# Experiment knobs
# ----------------------------
EXPERIMENT_NAME = "qwen2audio_eval_only_en_zh_bnb_int8_maxnew64"
RUN_BASELINE = True
RUN_INT8 = True
EVAL_SPLIT = "eval"   # "eval" or "dev"
USE_DEV_SMOKE = False
DEV_SMOKE_N = 16

# Registry filenames
SUMMARY_REGISTRY_CSV = REGISTRY_ROOT / "all_experiment_summaries.csv"
PREDICTIONS_INDEX_CSV = REGISTRY_ROOT / "all_prediction_files.csv"

print("project root exists:", PROJECT_ROOT.exists())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


# ============================================================
# CELL 2: Data and shared objects
# ============================================================

dev_df = pd.read_csv(DEV_MANIFEST).copy()
eval_df = pd.read_csv(EVAL_MANIFEST).copy()

dev_df["audio_path"] = dev_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))
eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

if EVAL_SPLIT == "eval":
    run_df = eval_df.copy()
elif EVAL_SPLIT == "dev":
    run_df = dev_df.copy()
else:
    raise ValueError(f"Unsupported EVAL_SPLIT={EVAL_SPLIT}")

if USE_DEV_SMOKE:
    run_df = dev_df.head(DEV_SMOKE_N).copy()

processor = AutoProcessor.from_pretrained(MODEL_ID)

comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("dev rows:", len(dev_df))
print("eval rows:", len(eval_df))
print("run split:", EVAL_SPLIT)
print("run rows:", len(run_df))


# ============================================================
# CELL 3: Helper functions only
# ============================================================

def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
    audio, sr = sf.read(str(audio_path))

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=text,
        audios=[audio],
        sampling_rate=sr,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(DEVICE) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

    pred = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return pred


def run_inference(model, df_input, run_name):
    preds = []
    start = time.time()

    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        pred = generate_translation(model, processor, Path(row["audio_path"]))
        preds.append(pred)

    elapsed = time.time() - start

    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
    comet_data = [
        {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
        for _, r in pred_df.iterrows()
    ]

    comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
    comet_score = comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]
    return comet_score


def cleanup_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


def make_experiment_dir(experiment_name: str) -> Path:
    exp_dir = RESULTS_ROOT / experiment_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    return exp_dir


def append_csv_row(csv_path: Path, row_df: pd.DataFrame):
    if csv_path.exists():
        old_df = pd.read_csv(csv_path)
        new_df = pd.concat([old_df, row_df], ignore_index=True)
    else:
        new_df = row_df.copy()
    new_df.to_csv(csv_path, index=False)


def save_run_artifacts(
    exp_dir: Path,
    experiment_name: str,
    run_name: str,
    split_name: str,
    pred_df: pd.DataFrame,
    seconds: float,
    comet_score: float,
    extra_meta: Dict,
):
    pred_path = exp_dir / f"{run_name}_preds.csv"
    pred_df.to_csv(pred_path, index=False, encoding="utf-8")

    summary_row = {
        "timestamp_utc": datetime.utcnow().isoformat(),
        "experiment_name": experiment_name,
        "run": run_name,
        "split": split_name,
        "rows": len(pred_df),
        "seconds": seconds,
        "seconds_per_item": seconds / max(len(pred_df), 1),
        "comet": comet_score,
        "model_id": MODEL_ID,
        "prompt_text": PROMPT_TEXT,
        "max_new_tokens": MAX_NEW_TOKENS,
        "comet_batch_size": COMET_BATCH_SIZE,
        "predictions_csv": str(pred_path),
    }
    summary_row.update(extra_meta)

    summary_df = pd.DataFrame([summary_row])

    run_summary_path = exp_dir / f"{run_name}_summary.csv"
    summary_df.to_csv(run_summary_path, index=False)

    append_csv_row(SUMMARY_REGISTRY_CSV, summary_df)

    pred_index_df = pd.DataFrame([{
        "timestamp_utc": summary_row["timestamp_utc"],
        "experiment_name": experiment_name,
        "run": run_name,
        "split": split_name,
        "predictions_csv": str(pred_path),
    }])
    append_csv_row(PREDICTIONS_INDEX_CSV, pred_index_df)

    meta_path = exp_dir / f"{run_name}_meta.json"
    with meta_path.open("w", encoding="utf-8") as f:
        json.dump(summary_row, f, indent=2, ensure_ascii=False)

    print(f"{run_name} predictions saved to:", pred_path)
    print(f"{run_name} summary saved to:", run_summary_path)
    print(f"{run_name} metadata saved to:", meta_path)
    return pred_path, run_summary_path


# ============================================================
# CELL 4: Experiment folder
# ============================================================

exp_dir = make_experiment_dir(EXPERIMENT_NAME)
print("experiment dir:", exp_dir)


# ============================================================
# CELL 5: Baseline run + COMET + cleanup
# ============================================================

run_summaries: List[Dict] = []

if RUN_BASELINE:
    baseline_model = Qwen2AudioForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    baseline_model.eval()

    baseline_df, baseline_time = run_inference(
        baseline_model,
        run_df,
        f"baseline_fp16_{EVAL_SPLIT}",
    )

    baseline_comet = compute_comet(baseline_df, comet_model)

    baseline_extra = {
        "quantization_method": "none",
        "precision": "fp16",
        "eval_split_source": EVAL_SPLIT,
    }

    save_run_artifacts(
        exp_dir=exp_dir,
        experiment_name=EXPERIMENT_NAME,
        run_name=f"baseline_fp16_{EVAL_SPLIT}",
        split_name=EVAL_SPLIT,
        pred_df=baseline_df,
        seconds=baseline_time,
        comet_score=baseline_comet,
        extra_meta=baseline_extra,
    )

    print("baseline seconds:", round(baseline_time, 2))
    print("baseline COMET:", baseline_comet)

    run_summaries.append({
        "run": f"baseline_fp16_{EVAL_SPLIT}",
        "split": EVAL_SPLIT,
        "rows": len(baseline_df),
        "seconds": baseline_time,
        "comet": baseline_comet,
        "quantization_method": "none",
        "precision": "fp16",
    })

    cleanup_model(baseline_model)


# ============================================================
# CELL 6: Int8 run + COMET + cleanup
# ============================================================

if RUN_INT8:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    quant_model = Qwen2AudioForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    quant_model.eval()

    quant_df, quant_time = run_inference(
        quant_model,
        run_df,
        f"quantized_int8_{EVAL_SPLIT}",
    )

    quant_comet = compute_comet(quant_df, comet_model)

    quant_extra = {
        "quantization_method": "bitsandbytes_int8",
        "precision": "int8",
        "eval_split_source": EVAL_SPLIT,
    }

    save_run_artifacts(
        exp_dir=exp_dir,
        experiment_name=EXPERIMENT_NAME,
        run_name=f"quantized_int8_{EVAL_SPLIT}",
        split_name=EVAL_SPLIT,
        pred_df=quant_df,
        seconds=quant_time,
        comet_score=quant_comet,
        extra_meta=quant_extra,
    )

    print("int8 seconds:", round(quant_time, 2))
    print("int8 COMET:", quant_comet)

    run_summaries.append({
        "run": f"quantized_int8_{EVAL_SPLIT}",
        "split": EVAL_SPLIT,
        "rows": len(quant_df),
        "seconds": quant_time,
        "comet": quant_comet,
        "quantization_method": "bitsandbytes_int8",
        "precision": "int8",
    })

    cleanup_model(quant_model)


# ============================================================
# CELL 7: Final per-experiment summary
# ============================================================

summary = pd.DataFrame(run_summaries)

baseline_run_name = f"baseline_fp16_{EVAL_SPLIT}"
if baseline_run_name in set(summary["run"]):
    baseline_score = summary.loc[summary["run"] == baseline_run_name, "comet"].iloc[0]
    baseline_time = summary.loc[summary["run"] == baseline_run_name, "seconds"].iloc[0]
    summary["comet_diff_vs_baseline"] = summary["comet"] - baseline_score
    summary["speedup_vs_baseline"] = baseline_time / summary["seconds"]
else:
    summary["comet_diff_vs_baseline"] = pd.NA
    summary["speedup_vs_baseline"] = pd.NA

experiment_summary_path = exp_dir / "summary_experiment.csv"
summary.to_csv(experiment_summary_path, index=False)

print("experiment summary saved to:", experiment_summary_path)
print(summary)


project root exists: True
cuda available: True
gpu: NVIDIA RTX A6000


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/lightning_fabric/utilities/cloud_io.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torc

dev rows: 468
eval rows: 416
run split: eval
run rows: 416
experiment dir: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64


We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


...

We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

quantized_int8_eval:   0%|          | 0/416 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
GP

quantized_int8_eval predictions saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64/quantized_int8_eval_preds.csv
quantized_int8_eval summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64/quantized_int8_eval_summary.csv
quantized_int8_eval metadata saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64/quantized_int8_eval_meta.json
int8 seconds: 1050.9
int8 COMET: 0.77519199395409
experiment summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/qwen2audio_eval_only_en_zh_bnb_int8_maxnew64/summary_experiment.csv
                   run split  rows      seconds     comet quantization_method  \
0   baseline_fp16_eval  eval   416   272.678346  0.779163                none   
1  quantized_int8_eval  eval   416  1050.8985

In [8]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from numcodecs import Quantize, FixedScaleOffset, Zstd


# ============================================================
# 1. Tiny synthetic model
# ============================================================
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 16, bias=False)
        self.fc2 = nn.Linear(16, 16, bias=False)
        self.fc3 = nn.Linear(16, 4, bias=False)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


# ============================================================
# 2. Safer codec pipeline
# ============================================================
class CodecPipeline:
    def __init__(self, filters, compressor=None, encoded_dtype="f4"):
        self.filters = filters
        self.compressor = compressor
        self.encoded_dtype = np.dtype(encoded_dtype)

    def encode(self, arr: np.ndarray):
        x = np.asarray(arr)

        for f in self.filters:
            x = f.encode(x)

        x = np.asarray(x, dtype=self.encoded_dtype)
        filtered_shape = x.shape
        filtered_bytes = x.tobytes(order="C")

        if self.compressor is not None:
            filtered_bytes = self.compressor.encode(filtered_bytes)

        return {
            "payload": filtered_bytes,
            "filtered_shape": filtered_shape,
            "filtered_dtype": self.encoded_dtype.str,
        }

    def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
        payload = encoded_obj["payload"]
        filtered_shape = tuple(encoded_obj["filtered_shape"])
        filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])

        if self.compressor is not None:
            payload = self.compressor.decode(payload)

        x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)

        for f in reversed(self.filters):
            x = f.decode(x)

        x = np.asarray(x, dtype=original_dtype).reshape(original_shape)
        return x


# ============================================================
# 3. Custom compressed linear layer
#    with cached decoded weight
# ============================================================
class CompressedLinear(nn.Module):
    def __init__(self, linear_layer: nn.Linear, pipeline: CodecPipeline, description: str):
        super().__init__()

        weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)

        self.weight_shape = weight.shape
        self.in_features = linear_layer.in_features
        self.out_features = linear_layer.out_features
        self.pipeline = pipeline
        self.description = description

        self.encoded_weight = self.pipeline.encode(weight)

        # cache
        self._cached_weight = None
        self._cached_device = None
        self._cached_dtype = None

    def _get_cached_weight(self, device, dtype):
        if (
            self._cached_weight is None
            or self._cached_device != device
            or self._cached_dtype != dtype
        ):
            decoded = self.pipeline.decode(
                self.encoded_weight,
                original_shape=self.weight_shape,
                original_dtype=np.float32,
            )
            self._cached_weight = torch.tensor(decoded, dtype=dtype, device=device)
            self._cached_device = device
            self._cached_dtype = dtype

        return self._cached_weight

    def forward(self, x):
        weight = self._get_cached_weight(x.device, x.dtype)
        return F.linear(x, weight)


# ============================================================
# 4. Compression policy
# ============================================================
def choose_pipeline_from_sensitivity(sensitivity: float):
    """
    Higher sensitivity -> milder compression
    Lower sensitivity  -> stronger compression
    """
    if sensitivity >= 0.66:
        pipeline = CodecPipeline(
            filters=[Quantize(digits=4, dtype="f4", astype="f2")],
            compressor=Zstd(level=3),
            encoded_dtype="f2",
        )
        desc = 'Quantize(digits=4, astype="f2") + Zstd'

    elif sensitivity >= 0.33:
        pipeline = CodecPipeline(
            filters=[Quantize(digits=2, dtype="f4", astype="f2")],
            compressor=Zstd(level=3),
            encoded_dtype="f2",
        )
        desc = 'Quantize(digits=2, astype="f2") + Zstd'

    else:
        pipeline = CodecPipeline(
            filters=[FixedScaleOffset(scale=100.0, offset=0.0, dtype="f4", astype="i1")],
            compressor=Zstd(level=3),
            encoded_dtype="i1",
        )
        desc = 'FixedScaleOffset(scale=100, astype="i1") + Zstd'

    return pipeline, desc


# ============================================================
# 5. Helpers to replace one layer or all layers
# ============================================================
def get_module_by_name(model: nn.Module, module_name: str):
    module = model
    for part in module_name.split("."):
        module = getattr(module, part)
    return module


def set_module_by_name(model: nn.Module, module_name: str, new_module: nn.Module):
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


def get_linear_layer_names(model: nn.Module):
    names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            names.append(name)
    return names


def compress_model_per_layer(model: nn.Module, sensitivity_map: dict):
    compressed_model = copy.deepcopy(model)

    for layer_name in get_linear_layer_names(compressed_model):
        layer = get_module_by_name(compressed_model, layer_name)
        sensitivity = sensitivity_map[layer_name]
        pipeline, desc = choose_pipeline_from_sensitivity(sensitivity)

        print(f"Layer: {layer_name}")
        print(f"  sensitivity = {sensitivity:.4f}")
        print(f"  pipeline    = {desc}")

        compressed_layer = CompressedLinear(layer, pipeline, desc)
        set_module_by_name(compressed_model, layer_name, compressed_layer)

    return compressed_model


# ============================================================
# 6. Automatic sensitivity estimation
#    compress one layer at a time, measure output drift
# ============================================================
def estimate_layer_sensitivity(model: nn.Module, validation_inputs: torch.Tensor):
    model = copy.deepcopy(model).eval()

    with torch.no_grad():
        baseline_output = model(validation_inputs)

    raw_scores = {}
    layer_names = get_linear_layer_names(model)

    for layer_name in layer_names:
        test_model = copy.deepcopy(model).eval()

        original_layer = get_module_by_name(test_model, layer_name)

        # Use a fixed medium compression for sensitivity probing
        probe_pipeline = CodecPipeline(
            filters=[Quantize(digits=2, dtype="f4", astype="f2")],
            compressor=Zstd(level=3),
            encoded_dtype="f2",
        )
        probe_desc = 'Probe: Quantize(digits=2, astype="f2") + Zstd'

        compressed_layer = CompressedLinear(original_layer, probe_pipeline, probe_desc)
        set_module_by_name(test_model, layer_name, compressed_layer)

        with torch.no_grad():
            test_output = test_model(validation_inputs)

        drift = torch.mean(torch.abs(baseline_output - test_output)).item()
        raw_scores[layer_name] = drift

    # Normalize to [0, 1], where 1 means more sensitive
    values = list(raw_scores.values())
    min_v = min(values)
    max_v = max(values)

    if max_v - min_v < 1e-12:
        normalized_scores = {k: 1.0 for k in raw_scores}
    else:
        normalized_scores = {
            k: (v - min_v) / (max_v - min_v)
            for k, v in raw_scores.items()
        }

    return raw_scores, normalized_scores


# ============================================================
# 7. Global baseline compression
# ============================================================
def compress_model_global(model: nn.Module, pipeline: CodecPipeline, description: str):
    compressed_model = copy.deepcopy(model)

    for layer_name in get_linear_layer_names(compressed_model):
        layer = get_module_by_name(compressed_model, layer_name)
        compressed_layer = CompressedLinear(layer, pipeline, description)
        set_module_by_name(compressed_model, layer_name, compressed_layer)

    return compressed_model


# ============================================================
# 8. Storage helpers
# ============================================================
def estimate_layer_storage_bytes(layer: nn.Module):
    if isinstance(layer, nn.Linear):
        return layer.weight.detach().cpu().numpy().nbytes
    elif isinstance(layer, CompressedLinear):
        return len(layer.encoded_weight["payload"])
    return 0


def print_model_storage_report(original_model: nn.Module, compressed_model: nn.Module):
    print("\n=== Storage report ===")
    total_original = 0
    total_compressed = 0

    original_layers = dict(original_model.named_modules())
    compressed_layers = dict(compressed_model.named_modules())

    for name, orig_layer in original_layers.items():
        comp_layer = compressed_layers.get(name, None)

        if isinstance(orig_layer, nn.Linear):
            orig_bytes = estimate_layer_storage_bytes(orig_layer)
            comp_bytes = estimate_layer_storage_bytes(comp_layer)
            total_original += orig_bytes
            total_compressed += comp_bytes

            ratio = orig_bytes / comp_bytes if comp_bytes > 0 else float("inf")
            desc = getattr(comp_layer, "description", "N/A")

            print(
                f"{name}: original={orig_bytes} bytes, "
                f"compressed={comp_bytes} bytes, "
                f"ratio={ratio:.2f}x, "
                f"method={desc}"
            )

    total_ratio = total_original / total_compressed if total_compressed > 0 else float("inf")
    print(
        f"TOTAL: original={total_original} bytes, "
        f"compressed={total_compressed} bytes, "
        f"ratio={total_ratio:.2f}x"
    )


# ============================================================
# 9. Evaluation helper
# ============================================================
def evaluate_models(original_model: nn.Module, compressed_model: nn.Module, x: torch.Tensor, title: str):
    with torch.no_grad():
        y_original = original_model(x)
        y_compressed = compressed_model(x)

    mae = torch.mean(torch.abs(y_original - y_compressed)).item()
    mse = torch.mean((y_original - y_compressed) ** 2).item()

    print(f"\n=== {title} ===")
    print("Original output:")
    print(y_original)
    print("\nCompressed output:")
    print(y_compressed)
    print(f"\nMAE: {mae:.6f}")
    print(f"MSE: {mse:.6f}")

    return mae, mse


# ============================================================
# 10. Demo
# ============================================================
if __name__ == "__main__":
    torch.manual_seed(42)
    np.random.seed(42)

    model = TinyMLP().eval()

    # Synthetic validation set used only for sensitivity estimation
    validation_inputs = torch.randn(32, 8)

    # Synthetic test batch for final comparison
    test_inputs = torch.randn(3, 8)

    # --------------------------------------------------------
    # A. Estimate sensitivity automatically
    # --------------------------------------------------------
    raw_scores, sensitivity_map = estimate_layer_sensitivity(model, validation_inputs)

    print("=== Raw layer drift scores ===")
    for k, v in raw_scores.items():
        print(f"{k}: {v:.6f}")

    print("\n=== Normalized sensitivity scores ===")
    for k, v in sensitivity_map.items():
        print(f"{k}: {v:.4f}")

    # --------------------------------------------------------
    # B. Adaptive compression
    # --------------------------------------------------------
    adaptive_model = compress_model_per_layer(model, sensitivity_map)

    # --------------------------------------------------------
    # C. Global baseline compression
    # --------------------------------------------------------
    global_pipeline = CodecPipeline(
        filters=[Quantize(digits=2, dtype="f4", astype="f2")],
        compressor=Zstd(level=3),
        encoded_dtype="f2",
    )
    global_desc = 'Global: Quantize(digits=2, astype="f2") + Zstd'
    global_model = compress_model_global(model, global_pipeline, global_desc)

    # --------------------------------------------------------
    # D. Evaluate adaptive model
    # --------------------------------------------------------
    evaluate_models(model, adaptive_model, test_inputs, title="Adaptive compression")
    print_model_storage_report(model, adaptive_model)

    # --------------------------------------------------------
    # E. Evaluate global baseline
    # --------------------------------------------------------
    evaluate_models(model, global_model, test_inputs, title="Global compression baseline")
    print_model_storage_report(model, global_model)

=== Raw layer drift scores ===
fc1: 0.000725
fc2: 0.000895
fc3: 0.000856

=== Normalized sensitivity scores ===
fc1: 0.0000
fc2: 1.0000
fc3: 0.7696
Layer: fc1
  sensitivity = 0.0000
  pipeline    = FixedScaleOffset(scale=100, astype="i1") + Zstd
Layer: fc2
  sensitivity = 1.0000
  pipeline    = Quantize(digits=4, astype="f2") + Zstd
Layer: fc3
  sensitivity = 0.7696
  pipeline    = Quantize(digits=4, astype="f2") + Zstd

=== Adaptive compression ===
Original output:
tensor([[ 0.0268,  0.0323,  0.0084, -0.1171],
        [ 0.0251, -0.0287, -0.0285, -0.0150],
        [ 0.0181, -0.0454,  0.0007,  0.0430]])

Compressed output:
tensor([[ 0.0262,  0.0332,  0.0071, -0.1160],
        [ 0.0258, -0.0283, -0.0276, -0.0174],
        [ 0.0165, -0.0453, -0.0012,  0.0437]])

MAE: 0.001036
MSE: 0.000001

=== Storage report ===
fc1: original=512 bytes, compressed=137 bytes, ratio=3.74x, method=FixedScaleOffset(scale=100, astype="i1") + Zstd
fc2: original=1024 bytes, compressed=522 bytes, ratio=1.96x, me